In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

df = pd.read_csv('master_dataset.csv')
df = df[df['Compound'] != 'INTERMEDIATE']
df = df[df['Compound'] != 'WET']
df = df[df['Compound'] != 'UNKNOWN']
print(df['Compound'].value_counts())

# fit LapTime ~ TyreLife + LapNumber separately per compound,
# then use the residual as the fuel/tyre corrected lap time
for compound in ['SOFT', 'MEDIUM', 'HARD']:
    subset = df[df['Compound'] == compound]
    subset = subset.dropna(subset=['LapTime', 'TyreLife', 'LapNumber'])
    X = subset[['TyreLife', 'LapNumber']]
    y = subset['LapTime']
    model = LinearRegression().fit(X, y)
    tyre_slope = model.coef_[0]
    lap_slope = model.coef_[1]
    df.loc[df['Compound'] == compound, 'CorrectedLapTime'] = subset['LapTime'] - (tyre_slope * subset['TyreLife']) - (lap_slope * subset['LapNumber'])
    print(compound, tyre_slope, lap_slope)

print(df[['LapTime', 'CorrectedLapTime']].head(10))
print(df['CorrectedLapTime'].isnull().sum())  # rows that couldn't be corrected (missing TyreLife/LapNumber)

df.to_csv('physics_adjusted_dataset.csv', index=False)
print("saved")

Compound
HARD      47966
MEDIUM    38977
SOFT      12864
Name: count, dtype: int64
SOFT -0.3213050950859463 -0.05620636394620768
MEDIUM -0.10337888394563151 -0.1388466498503789
HARD 0.0724113766763273 -0.24737917627499123
   LapTime  CorrectedLapTime
0   95.982         97.745490
1   96.169         97.932490
2   97.109         98.872490
3   98.294        101.522984
4   97.893        101.121984
5   97.571        100.799984
6   97.652        100.880984
7   97.046        100.274984
8   97.112        100.340984
9   97.852         99.305353
875
saved


In [2]:
print(df['Compound'].value_counts())
print(df.shape)

Compound
HARD      47966
MEDIUM    38977
SOFT      12864
Name: count, dtype: int64
(100486, 12)


In [3]:
print(df['Compound'].isna().sum())
print(df['Compound'].unique())

679
['MEDIUM' 'SOFT' 'HARD' nan]
